In [0]:
import re
from collections import defaultdict

# List all files in the customer incoming directory
files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/customer/")

# Group files by dataset name
grouped = defaultdict(list)
for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

# Function to extract timestamp from filename
def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

# Sort files and identify latest vs old files
file_info = {}
for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    file_info[dataset] = {
        'latest': sorted_files[0],
        'old_files': sorted_files[1:]
    }

print(f"Found {len(file_info)} dataset(s) to process:")
for dataset, info in file_info.items():
    print(f"  - {dataset}: 1 latest file, {len(info['old_files'])} old file(s)")

In [0]:
from pyspark.sql.functions import to_date

# Process each dataset
for dataset, info in file_info.items():
    latest_file = info['latest']
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Read the latest CSV file
    df_new = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(latest_file))
    
    # Convert date columns if they exist
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    # Create temp view for SQL operations
    temp_view = f"{dataset}_updates"
    df_new.createOrReplaceTempView(temp_view)
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Create new table
        spark.sql(f"""
            CREATE TABLE {table_name}
            USING DELTA
            AS SELECT * FROM {temp_view}
        """)
        record_count = spark.table(table_name).count()
        print(f"Created {table_name} with {record_count} records from {latest_file.split('/')[-1]}")
    else:
        # Use MERGE to upsert data - only update if values have changed
        result = spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {temp_view} AS source
            ON target.CustomerID = source.CustomerID
            WHEN MATCHED AND (
                NOT (target.CustomerName <=> source.CustomerName) OR
                NOT (target.Email <=> source.Email) OR
                NOT (target.City <=> source.City) OR
                NOT (target.Address <=> source.Address) OR
                NOT (target.LastUpdated <=> source.LastUpdated)
            ) THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        # Get metrics from merge result
        metrics = result.collect()[0]
        updated = metrics['num_updated_rows']
        inserted = metrics['num_inserted_rows']
        
        if updated > 0 or inserted > 0:
            print(f"Merged data from {latest_file.split('/')[-1]} into {table_name}: {updated} updated, {inserted} inserted")
        else:
            print(f"No changes detected from {latest_file.split('/')[-1]} - table {table_name} unchanged")

In [0]:
# Archive old files to the archive directory
for dataset, info in file_info.items():
    old_files = info['old_files']
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        archive_path = f"s3://retail-etl-project-revanth/archive/customer/{file_name}"
        dbutils.fs.mv(old_file, archive_path)
    
    if old_files:
        print(f"Archived {len(old_files)} old file(s) for {dataset}")
    else:
        print(f"No old files to archive for {dataset}")

In [0]:
# Read from Unity Catalog table
df = spark.table("retail_catalog.bronze.customers_raw")
print(f"Total customer records: {df.count()}")
display(df.limit(5))

In [0]:
import re
from collections import defaultdict

# List all files in the products incoming directory
files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/products/")

# Group files by dataset name
grouped = defaultdict(list)
for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

# Function to extract timestamp from filename
def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

# Sort files and identify latest vs old files
product_file_info = {}
for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    product_file_info[dataset] = {
        'latest': sorted_files[0],
        'old_files': sorted_files[1:]
    }

print(f"Found {len(product_file_info)} product dataset(s) to process:")
for dataset, info in product_file_info.items():
    print(f"  - {dataset}: 1 latest file, {len(info['old_files'])} old file(s)")

In [0]:
from pyspark.sql.functions import to_date

# Process each product dataset
for dataset, info in product_file_info.items():
    latest_file = info['latest']
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Read the latest CSV file
    df_new = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(latest_file))
    
    # Convert date columns if they exist
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    # Create temp view for SQL operations
    temp_view = f"{dataset}_updates"
    df_new.createOrReplaceTempView(temp_view)
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Create new table
        spark.sql(f"""
            CREATE TABLE {table_name}
            USING DELTA
            AS SELECT * FROM {temp_view}
        """)
        record_count = spark.table(table_name).count()
        print(f"Created {table_name} with {record_count} records from {latest_file.split('/')[-1]}")
    else:
        # Use MERGE to upsert data - only update if values have changed
        result = spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {temp_view} AS source
            ON target.ProductID = source.ProductID
            WHEN MATCHED AND (
                NOT (target.ProductName <=> source.ProductName) OR
                NOT (target.Category <=> source.Category) OR
                NOT (target.UnitPrice <=> source.UnitPrice)
            ) THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        # Get metrics from merge result
        metrics = result.collect()[0]
        updated = metrics['num_updated_rows']
        inserted = metrics['num_inserted_rows']
        
        if updated > 0 or inserted > 0:
            print(f"Merged data from {latest_file.split('/')[-1]} into {table_name}: {updated} updated, {inserted} inserted")
        else:
            print(f"No changes detected from {latest_file.split('/')[-1]} - table {table_name} unchanged")

In [0]:
# Archive old product files to the archive directory
for dataset, info in product_file_info.items():
    old_files = info['old_files']
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        archive_path = f"s3://retail-etl-project-revanth/archive/products/{file_name}"
        dbutils.fs.mv(old_file, archive_path)
    
    if old_files:
        print(f"Archived {len(old_files)} old file(s) for {dataset}")
    else:
        print(f"No old files to archive for {dataset}")

In [0]:
import re
from collections import defaultdict

# List all files in the sales incoming directory
files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/sales/")

# Group files by dataset name
grouped = defaultdict(list)
for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

# Function to extract timestamp from filename
def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

# Sort files and identify latest vs old files
sales_file_info = {}
for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    sales_file_info[dataset] = {
        'latest': sorted_files[0],
        'old_files': sorted_files[1:]
    }

print(f"Found {len(sales_file_info)} sales dataset(s) to process:")
for dataset, info in sales_file_info.items():
    print(f"  - {dataset}: 1 latest file, {len(info['old_files'])} old file(s)")

In [0]:
from pyspark.sql.functions import to_date, row_number
from pyspark.sql.window import Window

# Process each sales dataset
for dataset, info in sales_file_info.items():
    latest_file = info['latest']
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Read the latest CSV file
    df_new = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(latest_file))
    
    # Convert date columns if they exist
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    # Auto-detect key column for deduplication and MERGE
    key_col = None
    for col in df_new.columns:
        if col.lower() in ['salesid', 'transactionid', 'saleid', 'txnid']:
            key_col = col
            break
    
    if key_col:
        # Deduplicate source data by key column (keep first occurrence)
        window_spec = Window.partitionBy(key_col).orderBy(key_col)
        df_new = df_new.withColumn("row_num", row_number().over(window_spec)) \
                       .filter("row_num = 1") \
                       .drop("row_num")
    
    # Create temp view for SQL operations
    temp_view = f"{dataset}_updates"
    df_new.createOrReplaceTempView(temp_view)
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Create new table
        spark.sql(f"""
            CREATE TABLE {table_name}
            USING DELTA
            AS SELECT * FROM {temp_view}
        """)
        record_count = spark.table(table_name).count()
        print(f"Created {table_name} with {record_count} records from {latest_file.split('/')[-1]}")
    else:
        if key_col:
            # Use MERGE with detected key - only update if values have changed
            result = spark.sql(f"""
                MERGE INTO {table_name} AS target
                USING {temp_view} AS source
                ON target.{key_col} = source.{key_col}
                WHEN MATCHED AND (
                    NOT (target.CustomerID <=> source.CustomerID) OR
                    NOT (target.ProductID <=> source.ProductID) OR
                    NOT (target.StoreID <=> source.StoreID) OR
                    NOT (target.Quantity <=> source.Quantity) OR
                    NOT (target.TxnDate <=> source.TxnDate)
                ) THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            
            # Get metrics from merge result
            metrics = result.collect()[0]
            updated = metrics['num_updated_rows']
            inserted = metrics['num_inserted_rows']
            
            if updated > 0 or inserted > 0:
                print(f"Merged data from {latest_file.split('/')[-1]} into {table_name} using key {key_col}: {updated} updated, {inserted} inserted")
            else:
                print(f"No changes detected from {latest_file.split('/')[-1]} - table {table_name} unchanged")
        else:
            # No key found, append all records
            print(f"Warning: No unique key found for {table_name}, appending all records")
            df_new.write.mode("append").saveAsTable(table_name)

In [0]:
# Archive old sales files to the archive directory
for dataset, info in sales_file_info.items():
    old_files = info['old_files']
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        archive_path = f"s3://retail-etl-project-revanth/archive/sales/{file_name}"
        dbutils.fs.mv(old_file, archive_path)
    
    if old_files:
        print(f"Archived {len(old_files)} old file(s) for {dataset}")
    else:
        print(f"No old files to archive for {dataset}")

In [0]:
import re
from collections import defaultdict

# List all files in the stores incoming directory
files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/stores/")

# Group files by dataset name
grouped = defaultdict(list)
for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

# Function to extract timestamp from filename
def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

# Sort files and identify latest vs old files
stores_file_info = {}
for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    stores_file_info[dataset] = {
        'latest': sorted_files[0],
        'old_files': sorted_files[1:]
    }

print(f"Found {len(stores_file_info)} stores dataset(s) to process:")
for dataset, info in stores_file_info.items():
    print(f"  - {dataset}: 1 latest file, {len(info['old_files'])} old file(s)")

In [0]:
from pyspark.sql.functions import to_date

# Process each stores dataset
for dataset, info in stores_file_info.items():
    latest_file = info['latest']
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Read the latest CSV file
    df_new = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(latest_file))
    
    # Convert date columns if they exist
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    # Create temp view for SQL operations
    temp_view = f"{dataset}_updates"
    df_new.createOrReplaceTempView(temp_view)
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Create new table
        spark.sql(f"""
            CREATE TABLE {table_name}
            USING DELTA
            AS SELECT * FROM {temp_view}
        """)
        record_count = spark.table(table_name).count()
        print(f"Created {table_name} with {record_count} records from {latest_file.split('/')[-1]}")
    else:
        # Use MERGE to upsert data - only update if values have changed
        result = spark.sql(f"""
            MERGE INTO {table_name} AS target
            USING {temp_view} AS source
            ON target.StoreID = source.StoreID
            WHEN MATCHED AND (
                NOT (target.StoreName <=> source.StoreName) OR
                NOT (target.Region <=> source.Region)
            ) THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        # Get metrics from merge result
        metrics = result.collect()[0]
        updated = metrics['num_updated_rows']
        inserted = metrics['num_inserted_rows']
        
        if updated > 0 or inserted > 0:
            print(f"Merged data from {latest_file.split('/')[-1]} into {table_name}: {updated} updated, {inserted} inserted")
        else:
            print(f"No changes detected from {latest_file.split('/')[-1]} - table {table_name} unchanged")

In [0]:
# Archive old stores files to the archive directory
for dataset, info in stores_file_info.items():
    old_files = info['old_files']
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        archive_path = f"s3://retail-etl-project-revanth/archive/stores/{file_name}"
        dbutils.fs.mv(old_file, archive_path)
    
    if old_files:
        print(f"Archived {len(old_files)} old file(s) for {dataset}")
    else:
        print(f"No old files to archive for {dataset}")